##### Question 1:

Write a query using a CTE to calculate the total spent by each customer (customer_id), then find the overall average spending per customer across the dataset.

In [0]:
with cus_tot_spend as (
select ord.customer_id,sum(orditm.price) as tot_spend
from brazilian_e_commerce.sql_practice.olist_order_items_dataset orditm
join brazilian_e_commerce.sql_practice.olist_orders_dataset ord on orditm.order_id=ord.order_id
group by customer_id)
select round(avg(tot_spend),2) from cus_tot_spend;


##### Question 2:

Calculate the total revenue generated by each product category, and use the SUM() OVER() window function to display each category's total revenue alongside the percentage contribution to overall company revenue.

In [0]:
with revenue_for_each_category as (
select prod.product_category_name,round(sum(orditm.price),2) as total_revenue
from brazilian_e_commerce.sql_practice.olist_order_items_dataset orditm
join brazilian_e_commerce.sql_practice.olist_products_dataset prod 
on orditm.product_id=prod.product_id
group by prod.product_category_name
)
select product_category_name,total_revenue,
round((total_revenue / sum(total_revenue) over ()) *100,2) as percentage
from revenue_for_each_category
group by product_category_name,total_revenue
order by total_revenue desc;

In [0]:
select 
prod.product_category_name,
round(sum(orditm.price),2) as category_total,
round(
    (sum(orditm.price)/sum(sum(orditm.price)) over())*100,
    2
    ) as pct_category
from brazilian_e_commerce.sql_practice.olist_order_items_dataset orditm 
join brazilian_e_commerce.sql_practice.olist_products_dataset prod 
on orditm.product_id=prod.product_id
group by product_category_name
order by category_total desc;

##### Question 3:

For each order in olist_order_items_dataset, display the order_id, product_id, price, and use a window function to display the average price of all items within that same order.

In [0]:
select order_id,product_id,price,round(avg(price) over (partition by order_id),2) avg_order_price
from brazilian_e_commerce.sql_practice.olist_order_items_dataset;

## Part 2 (CTEs & Window Functions)

#####Question 4 (Ranking Window Functions):

Rank the top 3 most expensive products within each product_category_name using the DENSE_RANK() window function. Display the category name, product ID, price, and rank.

In [0]:
with product_ranks as (
select distinct prod.product_category_name,prod.product_id,orditm.price,
dense_rank() over(partition by prod.product_category_name order by orditm.price desc) as dense_rank
from brazilian_e_commerce.sql_practice.olist_order_items_dataset orditm
join brazilian_e_commerce.sql_practice.olist_products_dataset prod on
orditm.product_id=prod.product_id
where prod.product_category_name is not null)
select product_category_name,product_id,price,dense_rank
from product_ranks
where dense_rank <=3 
order by product_category_name,dense_rank;


##### Question 5 (Multi-stage CTE Aggregation):

Write a query using a CTE to calculate the total freight cost (SUM(freight_value)) spent by each customer, then filter the CTE in your main query to show only customers whose total freight spend is higher than $100.

In [0]:
with total_freight_cost as(
select c.customer_id, round(sum(oi.freight_value),2) total_freight
from brazilian_e_commerce.sql_practice.olist_order_items_dataset oi
join brazilian_e_commerce.sql_practice.olist_orders_dataset o on oi.order_id = o.order_id
join brazilian_e_commerce.sql_practice.olist_customers_dataset c on o.customer_id = c.customer_id
group by c.customer_id
order by total_freight desc)
select * from total_freight_cost 
where total_freight > 100;

##### Question 6 (Cumulative / Running Total):

For each order item, display the order_id, price, and calculate a running total of item prices ordered by order_id across the entire olist_order_items_dataset using SUM() OVER(ORDER BY ...).

In [0]:
select order_id,price,
round(sum(price) over(order by order_id),2) as running_total
from brazilian_e_commerce.sql_practice.olist_order_items_dataset;